# Standardizing Videos

Before we can extract motion features, the videos need to be put into a common format. This notebook takes the manually edited trial videos and writes standardized versions with the same frame rate, image size, and brightness normalization.

This step matters because later notebooks compare movement across videos. If videos differ in size, frame rate, or lighting, the extracted motion features can reflect recording differences instead of behavior.


## Configure Settings

These settings define the target video format.

- `TARGET_FPS`: how many frames per second to save.
- `SCALE`: how much to resize the manually edited video.
- `NORM_MODE`: how to reduce frame-to-frame lighting differences.
- Debug options let us process one short video while testing the code.


In [1]:
# Debug controls
TEST_SHORT_VIDEO = False
max_n_frames = 50 if TEST_SHORT_VIDEO else None
TEST_ONE_VID_IDX = None

# Target standardized-video settings
TARGET_FPS = 30
SCALE = 0.75
TARGET_WIDTH = round(1080 * SCALE)
TARGET_HEIGHT = round(1080 * SCALE)
NORM_MODE = "zscore"  # "nonorm", "zscore", "clahe"
PREPROC_VERSION = "v1.1.0"

In [2]:
# IMPORTS ----
from dotenv import load_dotenv
from pathlib import Path
import json
from datetime import datetime, timezone
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import LoadVideo as lv
load_dotenv()


# PATHS ----
DATA_ROOT = lv.get_data_root()
VIDEO_IDS = lv.list_video_ids(DATA_ROOT)

## Helper Functions

The helper functions below do two jobs: check that each input video is readable, then write a standardized output video.


### Validate Input Videos

Before processing a video, we check basic properties such as frame rate, image size, frame count, and duration. This catches path or formatting problems before the main loop runs.


In [3]:
def validate_video(in_path: Path, out_dir: Path):
    cap = cv2.VideoCapture(str(in_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {in_path}")

    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = frame_count / orig_fps if orig_fps > 0 else None
    print(f"{in_path.name} | FPS: {orig_fps} | Size: {orig_w}x{orig_h} | Frames: {frame_count} | Duration: {duration_sec:.2f}s")

    out_dir.mkdir(parents=True, exist_ok=True)
    assert TARGET_WIDTH == round(orig_w * SCALE), f"TARGET_WIDTH mismatch: {TARGET_WIDTH} vs {round(orig_w * SCALE)}"
    assert TARGET_HEIGHT == round(orig_h * SCALE), f"TARGET_HEIGHT mismatch: {TARGET_HEIGHT} vs {round(orig_h * SCALE)}"
    cap.release()
    cv2.destroyAllWindows()
    return True



### Normalize and Resample Frames

The main standardization function samples frames at the target frame rate, resizes each frame, normalizes the colors, and writes the result as a new MP4 file.

The important computer vision idea is that a video is just a sequence of images. Here, we make those images more comparable across trials before measuring motion.


In [4]:
# Per-frame z-score normalization reduces lighting differences across trials.
def _zscore_rescale(channel: np.ndarray) -> np.ndarray:
    g = channel.astype(np.float32)
    mu, sd = g.mean(), g.std()
    if sd < 1e-6:
        return channel
    z = (g - mu) / sd
    z = np.clip((z - z.min()) / (z.max() - z.min() + 1e-6) * 255.0, 0, 255)
    return z.astype(np.uint8)


def normalize_color_frame(frame: np.ndarray, mode: str) -> np.ndarray:
    if mode == "clahe":
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l_chan, a_chan, b_chan = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_chan = clahe.apply(l_chan)
        lab = cv2.merge([l_chan, a_chan, b_chan])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    if mode == "zscore":
        out = np.empty_like(frame)
        for c in range(3):
            out[:, :, c] = _zscore_rescale(frame[:, :, c])
        return out

    raise ValueError(f"Unknown NORM_MODE: {mode}")


def standardize_video(
    in_path: Path,
    out_path: Path,
    max_frames: int = None,
    video_id: str | None = None,
):
    validate_video(in_path, out_path.parent)
    cap = cv2.VideoCapture(str(in_path))

    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = frame_count / orig_fps if orig_fps > 0 else None

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, TARGET_FPS, (TARGET_WIDTH, TARGET_HEIGHT), isColor=True)

    # Sample frames at fixed time steps so each output video uses the same FPS.
    t = 0.0
    dt = 1.0 / TARGET_FPS
    processed_frames = 0

    total_frames = int(duration_sec * TARGET_FPS) if duration_sec else frame_count
    total_frames = min(total_frames, max_frames) if max_frames else total_frames

    for _ in tqdm(range(total_frames), desc="Processing frames"):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)

        color = normalize_color_frame(frame, NORM_MODE)
        writer.write(color)

        t += dt
        processed_frames += 1
        if max_frames and processed_frames >= max_frames:
            break

    cap.release()
    writer.release()
    return {
        "video_id": video_id,
        "input_file": in_path.name,
        "output_file": out_path.name,
        "input_path": str(in_path),
        "output_path": str(out_path),
        "orig_fps": orig_fps,
        "orig_width": orig_w,
        "orig_height": orig_h,
        "orig_frame_count": frame_count,
        "orig_duration_sec": duration_sec,
        "target_fps": TARGET_FPS,
        "target_width": TARGET_WIDTH,
        "target_height": TARGET_HEIGHT,
        "scale": SCALE,
        "scale_percent": lv.scale_to_int(SCALE),
        "norm_mode": NORM_MODE,
        "preproc_version": PREPROC_VERSION,
    }



## Process All Videos

The final cell loops over every video folder, standardizes the manually edited video, and writes a small log file. Those logs make the preprocessing step easier to audit later.


In [5]:
def main():
    video_ids = VIDEO_IDS
    assert len(video_ids) > 0, f"No video folders found in: {DATA_ROOT}"

    if TEST_ONE_VID_IDX is not None:
        assert 0 <= TEST_ONE_VID_IDX < len(video_ids)
        video_ids = [video_ids[TEST_ONE_VID_IDX]]

    rows = []
    for video_id in tqdm(video_ids, desc="Standardizing Videos"):
        video_dir = lv.get_video_dir(video_id, video_root=DATA_ROOT, require_exist=True)

        raw_path = lv.get_video_file(video_id, kind="manual", video_root=DATA_ROOT)
        out_path = lv.get_video_file(video_id, kind="standardized", video_root=DATA_ROOT, require_exists=False)

        row = standardize_video(raw_path, out_path, max_frames=max_n_frames, video_id=video_id)
        rows.append(row)

        log_payload = {
            **row,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }
        log_path = video_dir / "log_preproc.json"
        log_path.write_text(json.dumps(log_payload, indent=2), encoding="utf-8")

    if rows:
        log = pd.DataFrame(rows)
        log_path = DATA_ROOT / "preprocessing_log.csv"
        log.to_csv(log_path, index=False)
        print(f"\nDone.\nWrote standardized videos and per-video logs.\nSummary log: {log_path}")
    else:
        print("\nNo videos processed.")


if __name__ == "__main__":
    main()


Standardizing Videos:   0%|          | 0/27 [00:00<?, ?it/s]

manedit.mp4 | FPS: 30.0 | Size: 1080x1080 | Frames: 979 | Duration: 32.63s


Standardizing Videos:   4%|▎         | 1/27 [01:33<40:36, 93.71s/it]

manedit.mp4 | FPS: 30.0 | Size: 1080x1080 | Frames: 1541 | Duration: 51.37s


Standardizing Videos:   7%|▋         | 2/27 [03:53<50:17, 120.72s/it]

...

manedit.mp4 | FPS: 30.148383777980534 | Size: 1080x1080 | Frames: 303 | Duration: 10.05s


Standardizing Videos:  96%|█████████▋| 26/27 [16:51<00:32, 32.40s/it]

manedit.mp4 | FPS: 30.184382600442532 | Size: 1080x1080 | Frames: 302 | Duration: 10.01s


Standardizing Videos: 100%|██████████| 27/27 [17:24<00:00, 38.70s/it]


Done.
Wrote standardized videos and per-video logs.
Summary log: G:\My Drive\TEACH_lumier_mledu\Repo\Data\preprocessing_log.csv
